# Classificação — EDA e Baseline

**Objetivo:** Prever uma classe/categoria a partir das variáveis de entrada.

Preencha o caminho do dataset em `CAMINHO_DADOS` e vá rodando célula por célula.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 2. Carregar os dados

In [ ]:
# Ajuste para o seu arquivo (coloque o CSV em ../data/)
CAMINHO_DADOS = '../data/seu_dataset.csv'

df = pd.read_csv(CAMINHO_DADOS)
print('Linhas x Colunas:', df.shape)
df.head()

## 3. Visão geral e qualidade dos dados

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Valores nulos por coluna
nulos = df.isnull().sum().sort_values(ascending=False)
nulos[nulos > 0]

In [ ]:
# Linhas duplicadas
print('Duplicadas:', df.duplicated().sum())

## 4. Distribuições

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
df[num_cols].hist(figsize=(14, 10), bins=30)
plt.tight_layout(); plt.show()

## 5. Correlações

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlação'); plt.show()

## 6. Variável alvo (classe)

In [ ]:
ALVO = 'target'  # <- nome da coluna alvo
df[ALVO].value_counts(normalize=True).plot(kind='bar')
plt.title('Distribuição das classes'); plt.show()

## 7. Baseline — split, pipeline e modelos

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X = df.drop(columns=[ALVO])
y = df[ALVO]

# IMPORTANTE: split ANTES de escalar (evita data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
modelos = {
    'LogReg': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(random_state=42),
}
for nome, mod in modelos.items():
    mod.fit(X_train_s, y_train)
    pred = mod.predict(X_test_s)
    print(f'\n===== {nome} =====')
    print(classification_report(y_test, pred))

## 8. Próximos passos
- Matriz de confusão do melhor modelo
- Tuning com GridSearchCV
- Testar XGBoost
- Salvar modelo em ../models/ com joblib